# ELM Validation: Benchmark Results

12 LLMs evaluated on 31 ELM-CPG pairs (15 valid, 16 invalid).

In [1]:
import pandas as pd, numpy as np, os, warnings
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'results')
for d in ['../results', 'results']:
    if os.path.isdir(d): RESULTS_DIR = d; break

frames = []
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith('results-') and f.endswith('.csv') and 'v1_' not in RESULTS_DIR:
        path = os.path.join(RESULTS_DIR, f)
        if not os.path.isfile(path): continue
        df = pd.read_csv(path)
        for col in ['valid','correct','expected_valid']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip().str.lower() == 'true'
        df['case'] = df['file'].str.replace('.json','',regex=False)
        frames.append(df)
data = pd.concat(frames, ignore_index=True)
n_models = data['model'].nunique()
n_cases = data['file'].nunique()
n_valid = int(data[data['model']==data['model'].unique()[0]]['expected_valid'].sum())
n_invalid = n_cases - n_valid
print(f"Loaded {len(data)} obs: {n_models} models x {n_cases} cases ({n_valid} valid, {n_invalid} invalid)")
print(f"Base rate: {n_valid/n_cases:.1%}")

Loaded 372 obs: 12 models x 31 cases (15 valid, 16 invalid)
Base rate: 48.4%


## Per-Model Accuracy with Wilson 95% CIs

In [2]:
MODEL_ORDER = ['gpt-oss-20b','gpt-oss-120b','qwen3-32b','llama-3.3-70b',
               'medgemma-4b','medgemma-1.5-4b','gemma-3-4b','phi-3-mini',
               'qwen-2.5-3b','llama-3.2-3b','qwen-2.5-1.5b','llama-3.2-1b']
rows = []
for model in MODEL_ORDER:
    mdf = data[data['model']==model]
    if len(mdf)==0: continue
    n=len(mdf); k=int(mdf['correct'].sum()); acc=k/n
    ci_lo,ci_hi = proportion_confint(k,n,alpha=0.05,method='wilson')
    tp=int((mdf['valid']&mdf['expected_valid']).sum())
    fp=int((mdf['valid']&~mdf['expected_valid']).sum())
    tn=int((~mdf['valid']&~mdf['expected_valid']).sum())
    fn=int((~mdf['valid']&mdf['expected_valid']).sum())
    sens=tp/(tp+fn) if (tp+fn) else 0; spec=tn/(tn+fp) if (tn+fp) else 0
    f1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0
    rows.append({'Model':model,'Correct':f'{k}/{n}','Accuracy':f'{acc:.1%}',
                 '95% CI':f'[{ci_lo:.3f}, {ci_hi:.3f}]',
                 'Sens':f'{sens:.2f}','Spec':f'{spec:.2f}','F1':f'{f1:.2f}'})
pd.DataFrame(rows)

,Model,Correct,Accuracy,95% CI,Sens,Spec,F1
0,gpt-oss-20b,29/31,93.5%,"[0.793, 0.982]",1.00,0.88,0.94
1,gpt-oss-120b,27/31,87.1%,"[0.711, 0.949]",0.87,0.88,0.87
2,qwen3-32b,28/31,90.3%,"[0.751, 0.967]",0.93,0.88,0.90
3,llama-3.3-70b,27/31,87.1%,"[0.711, 0.949]",0.93,0.81,0.88
4,medgemma-4b,15/31,48.4%,"[0.320, 0.652]",1.00,0.00,0.65
5,medgemma-1.5-4b,15/31,48.4%,"[0.320, 0.652]",1.00,0.00,0.65
6,gemma-3-4b,15/31,48.4%,"[0.320, 0.652]",1.00,0.00,0.65
7,phi-3-mini,17/31,54.8%,"[0.378, 0.708]",0.93,0.19,0.67
8,qwen-2.5-3b,19/31,61.3%,"[0.438, 0.763]",1.00,0.25,0.71
9,llama-3.2-3b,14/31,45.2%,"[0.292, 0.622]",0.87,0.06,0.60


## Fisher's Exact Test: Frontier vs Small

In [3]:
large = ['gpt-oss-120b','llama-3.3-70b','qwen3-32b','gpt-oss-20b']
small = [m for m in data['model'].unique() if m not in large]
ldf = data[data['model'].isin(large)]; sdf = data[data['model'].isin(small)]
lc,lt = int(ldf['correct'].sum()),len(ldf)
sc,st = int(sdf['correct'].sum()),len(sdf)
odds,p = stats.fisher_exact([[lc,lt-lc],[sc,st-sc]])
print(f"Frontier: {lc}/{lt} ({lc/lt:.1%})")
print(f"Small:    {sc}/{st} ({sc/st:.1%})")
print(f"Fisher exact: OR={odds:.2f}, p={p:.2e}")

Frontier: 111/124 (89.5%)
Small:    129/248 (52.0%)
Fisher exact: OR=7.88, p=1.13e-13


## McNemar Pairwise Tests (Bonferroni corrected)

In [4]:
from itertools import combinations
functioning = [m for m in MODEL_ORDER if m in data['model'].unique()]
pairs = list(combinations(functioning, 2))
alpha_corr = 0.05/len(pairs)
print(f"{len(pairs)} pairs, Bonferroni alpha={alpha_corr:.5f}\n")
sig_pairs = []
for m1,m2 in pairs:
    d1 = data[data['model']==m1].set_index('file')['correct']
    d2 = data[data['model']==m2].set_index('file')['correct']
    common = d1.index.intersection(d2.index)
    b = int((d1[common]&~d2[common]).sum()); c = int((~d1[common]&d2[common]).sum())
    nd = b+c
    p = 1.0 if nd==0 else stats.binomtest(max(b,c),nd,0.5).pvalue
    if p < 0.05: sig_pairs.append({'Pair':f'{m1} vs {m2}','b':b,'c':c,'p':f'{p:.4f}',
                                    'Sig':'**' if p<alpha_corr else '*'})
if sig_pairs:
    print("Pairs reaching uncorrected significance (p<0.05):")
    display(pd.DataFrame(sig_pairs))
else:
    print("No pairs reached p<0.05")

66 pairs, Bonferroni alpha=0.00076

Pairs reaching uncorrected significance (p<0.05):


,Pair,b,c,p,Sig
0,gpt-oss-20b vs medgemma-4b,14,0,0.0001,**
1,gpt-oss-20b vs medgemma-1.5-4b,14,0,0.0001,**
2,gpt-oss-20b vs gemma-3-4b,14,0,0.0001,**
3,gpt-oss-20b vs phi-3-mini,12,0,0.0005,**
4,gpt-oss-20b vs qwen-2.5-3b,10,0,0.0020,*
5,gpt-oss-20b vs llama-3.2-3b,15,0,0.0001,**
6,gpt-oss-20b vs qwen-2.5-1.5b,15,2,0.0023,*
7,gpt-oss-20b vs llama-3.2-1b,11,0,0.0010,*
8,gpt-oss-120b vs medgemma-4b,14,2,0.0042,*
9,gpt-oss-120b vs medgemma-1.5-4b,14,2,0.0042,*
